# 06 — Training, saving and reloading tokenizers (Parts E and F)

This notebook has two phases:

1. **Build it here** — 🧩 exercises where you write the Hugging Face code *step by step* (model → pre-tokenizer → decoder → trainer → train → inspect → save → wrap).
2. **Move it into `src/`** — you then package the same code as functions in `src/train_bpe.py`, `src/train_wordpiece.py`, `src/tokenizer_io.py` and make the tests pass; the experiments at the end of this notebook use those functions.

Works offline.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
%load_ext autoreload
%autoreload 2

CORPUS = ROOT / "data/processed/train_corpus.txt"
lines = CORPUS.read_text(encoding="utf-8").splitlines()
print(len(lines), "training sentences")

## The training corpus (document it in your report!)

In [ ]:
import re
def script(line): return "arabic" if re.search(r"[\u0600-\u06FF]", line) else "latin"
print(pd.Series([script(l) for l in lines]).value_counts().to_dict())
print("en:", len((ROOT / "data/processed/train_en.txt").read_text(encoding="utf-8").splitlines()),
      "| fr:", len((ROOT / "data/processed/train_fr.txt").read_text(encoding="utf-8").splitlines()))
print("avg characters per sentence:", round(sum(map(len, lines)) / len(lines), 1)); print(lines[:2])

## Part E, task 7 — byte-level BPE, step by step

Recall the pipeline: normalizer → pre-tokenizer → **model** → post-processor → ids, and ids → **decoder** → text.

### 🧩 Exercise 6.1 — Assemble the (untrained) BPE tokenizer

Create a `Tokenizer` whose model is BPE, then attach a **byte-level pre-tokenizer** and the matching **decoder**.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Imports are given. Both `ByteLevel` objects have the same name but live in different modules: the decoder is imported as `ByteLevelDecoder`.
* Model: `BPE(unk_token="[UNK]")` is passed to `Tokenizer(...)`.
* Pre-tokenizer: `ByteLevel(add_prefix_space=False)`; decoder: `ByteLevelDecoder()`. Assign them to `tokenizer.pre_tokenizer` and `tokenizer.decoder`.
* Docs: https://huggingface.co/docs/tokenizers/components

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

unk_token = "[UNK]"

tokenizer = Tokenizer(____)
tokenizer.pre_tokenizer = ____
tokenizer.decoder = ____

In [ ]:
assert type(tokenizer.model).__name__ == "BPE"
assert type(tokenizer.pre_tokenizer).__name__ == "ByteLevel"
assert type(tokenizer.decoder).__name__ == "ByteLevel"
print("✅ Exercise 6.1 OK")

### 🧩 Exercise 6.2 — Configure the trainer

Create a `BpeTrainer` with a **2 000** target vocabulary, `min_frequency` 2, the four special tokens `[UNK] [PAD] [BOS] [EOS]` (in this order) and the **256-byte initial alphabet**.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `BpeTrainer(vocab_size=..., min_frequency=..., special_tokens=[...], initial_alphabet=..., show_progress=False)`.
* The byte alphabet is provided by the pre-tokenizer class: `ByteLevel.alphabet()` (a list of 256 symbols).
* Special tokens get ids 0, 1, 2, 3 in the order of the list. Why does `[UNK]` come first here?

In [ ]:
from tokenizers.trainers import BpeTrainer

trainer = BpeTrainer(
    vocab_size=____,
    min_frequency=____,
    special_tokens=____,
    initial_alphabet=____,
    show_progress=False,
)

In [ ]:
assert len(ByteLevel.alphabet()) == 256
print("✅ Exercise 6.2 OK")

### 🧩 Exercise 6.3 — Train, then inspect

Train on the corpus **file**, then encode a sentence and read `.tokens`, `.ids` and `.offsets`; decode the ids; look up a special token's id.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `tokenizer.train(files, trainer)` — `files` is a **list of strings** (paths).
* `tokenizer.encode(text)` returns an `Encoding` with attributes `tokens`, `ids`, `offsets`.
* `tokenizer.decode(ids)` reconstructs text; `tokenizer.token_to_id("[PAD]")` gives a special token's id.
* `tokenizer.get_vocab_size()` is the final size.

In [ ]:
tokenizer.____([str(CORPUS)], ____)

print("vocab size:", tokenizer.____())
enc = tokenizer.____("Les modèles de langage, c'est génial !")
print(enc.____); print(enc.____); print(enc.____)
print(repr(tokenizer.____(enc.ids)))
print("[PAD] id:", tokenizer.____("[PAD]"))

In [ ]:
assert tokenizer.get_vocab_size() == 2000
assert [tokenizer.token_to_id(t) for t in ["[UNK]", "[PAD]", "[BOS]", "[EOS]"]] == [0, 1, 2, 3]
assert tokenizer.decode(enc.ids) == "Les modèles de langage, c'est génial !"
print("✅ Exercise 6.3 OK")

**Look at the result** before moving on: which tokens look like whole words, which are single bytes? What does `Ġ` mean? Why did `è` become two characters?

## Part E, task 8 — WordPiece, step by step

### 🧩 Exercise 6.4 — Assemble the WordPiece tokenizer

Same idea as for BPE, with **four** components: WordPiece model, BERT normalizer (lowercase + strip accents), BERT pre-tokenizer, and the WordPiece decoder with prefix `##`.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `WordPiece(unk_token="[UNK]")`, `BertNormalizer(lowercase=True, strip_accents=True)`, `BertPreTokenizer()`, `WordPieceDecoder(prefix="##")`.
* Assign to `.normalizer`, `.pre_tokenizer`, `.decoder` of the tokenizer.
* Docs: https://huggingface.co/docs/tokenizers/components

In [ ]:
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder

my_wp = Tokenizer(____)
my_wp.normalizer = ____
my_wp.pre_tokenizer = ____
my_wp.decoder = ____

In [ ]:
assert [type(x).__name__ for x in (my_wp.model, my_wp.normalizer, my_wp.pre_tokenizer, my_wp.decoder)] == ["WordPiece", "BertNormalizer", "BertPreTokenizer", "WordPiece"]
print("✅ Exercise 6.4 OK")

### 🧩 Exercise 6.5 — WordPiece trainer and training

Create a `WordPieceTrainer` (2 000, `min_frequency` 2, special tokens `[UNK] [PAD] [CLS] [SEP] [MASK]`, continuation prefix `##`) and train from an **iterator of strings** this time.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `WordPieceTrainer(vocab_size=..., min_frequency=..., special_tokens=[...], continuing_subword_prefix="##", show_progress=False)`.
* `tokenizer.train_from_iterator(iterator, trainer=trainer)` — no files needed: `lines` is already in memory.

In [ ]:
from tokenizers.trainers import WordPieceTrainer

wp_trainer = ____(
    vocab_size=2000,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    continuing_subword_prefix=____,
    show_progress=False,
)
my_wp.____(lines, trainer=wp_trainer)
print("vocab size:", my_wp.get_vocab_size())
enc = my_wp.encode("Unbelievably, the Café is incomprehensible!")
print(enc.tokens); print(repr(my_wp.decode(enc.ids)))

In [ ]:
assert my_wp.get_vocab_size() == 2000
assert any(t.startswith("##") for t in enc.tokens)
assert my_wp.decode(enc.ids).startswith("unbelievably")
print("✅ Exercise 6.5 OK")

### 🧩 Exercise 6.6 — Post-processor: add `[CLS]` and `[SEP]`

BERT-style models expect `[CLS] sentence [SEP]`. Add a template post-processor to a **copy** of your WordPiece tokenizer and see what `encode` does now.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `TemplateProcessing(single="[CLS] $A [SEP]", special_tokens=[("[CLS]", id), ("[SEP]", id)])` — `$A` stands for the sentence.
* Ids of the special tokens: `tokenizer.token_to_id(...)`.
* Assign it to `.post_processor`. `Tokenizer.from_str(my_wp.to_str())` makes an independent copy.
* Remember notebook 02: `encode(..., add_special_tokens=False)` switches this off.

In [ ]:
from tokenizers.processors import TemplateProcessing

wp_bert = Tokenizer.from_str(my_wp.to_str())          # an independent copy
wp_bert.post_processor = ____(
    single=____,
    special_tokens=[("[CLS]", wp_bert.token_to_id("[CLS]")), ("[SEP]", ____)],
)
print(wp_bert.encode("hello world").tokens)
print(wp_bert.encode("hello world", add_special_tokens=False).tokens)

In [ ]:
assert wp_bert.encode("hello world").tokens[0] == "[CLS]" and wp_bert.encode("hello world").tokens[-1] == "[SEP]"
assert wp_bert.encode("hello world", add_special_tokens=False).tokens[0] != "[CLS]"
print("✅ Exercise 6.6 OK")

## ⏸ Stop: package your code in `src/` now

You now know every line. Turn the exercises into reusable functions:

| You wrote in… | → package it in |
|---|---|
| exercises 6.1–6.3 (BPE pipeline, trainer, training) | `src/train_bpe.py` → `build_bpe_tokenizer`, `make_trainer`, `train_bpe`, `train_bpe_from_texts` |
| exercises 6.4–6.5 (WordPiece pipeline, trainer, training) | `src/train_wordpiece.py` |

Then run `pytest tests/test_bpe.py tests/test_wordpiece.py`. When both are green, continue below: the rest of the notebook uses **your functions**.

In [ ]:
from src.train_bpe import train_bpe, train_bpe_from_texts
from src.train_wordpiece import train_wordpiece
from src.tokenizer_io import save_tokenizer, load_tokenizer, to_transformers, find_mismatches
from tests.conftest import SENTENCES

bpe = train_bpe(CORPUS, vocab_size=2000, min_frequency=2)
wp = train_wordpiece(CORPUS, vocab_size=2000, min_frequency=2)
print("BPE:", bpe.get_vocab_size(), "| WordPiece:", wp.get_vocab_size())
by_id = sorted(bpe.get_vocab().items(), key=lambda kv: kv[1])
print("BPE first 10:", by_id[:10]); print("BPE tokens 260-290:", [t for t, _ in by_id[260:290]]); print("BPE last 20:", [t for t, _ in by_id[-20:]])

In [ ]:
tests = ["Hello, how are you?", "L'apprentissage automatique est passionnant.", "الذكاء الاصطناعي", "Python 3.12 🙂"]
for s in tests:
    e1, e2 = bpe.encode(s), wp.encode(s)
    print(s); print("  BPE      :", e1.tokens, "| lossless:", bpe.decode(e1.ids) == s)
    print("  WordPiece:", e2.tokens, "|", repr(wp.decode(e2.ids)))

### Requested vs. final vocabulary size, and `min_frequency`

In [ ]:
rows = []
for vs in [500, 1000, 2000, 5000, 10000]:
    for mf in [1, 2, 10]:
        rows.append({"requested": vs, "min_frequency": mf, "final": train_bpe(CORPUS, vocab_size=vs, min_frequency=mf).get_vocab_size()})
pd.DataFrame(rows).pivot(index="requested", columns="min_frequency", values="final")

**Questions (BPE)** — 1. role of `BPE`, `BpeTrainer`, `ByteLevel`? 2. pre-tokenizer vs decoder? 3. what changes with `min_frequency`? 4. when is final ≠ requested size (look at the table)? 5. why `initial_alphabet=ByteLevel.alphabet()`?

**Your answer:**

_(write here)_

In [ ]:
cmp = pd.DataFrame({"sentence": SENTENCES,
                    "bpe_tokens": [len(bpe.encode(s).tokens) for s in SENTENCES],
                    "wordpiece_tokens": [len(wp.encode(s).tokens) for s in SENTENCES],
                    "wordpiece_unk": [wp.encode(s).tokens.count("[UNK]") for s in SENTENCES]})
cmp

**Questions (WordPiece)** — 1. BPE vs WordPiece? 2. role of `BertNormalizer`? 3. why `##`? 4. why is the decoder important? 5. why the same corpus for a fair comparison? 6. Look at the Arabic sentences in the table: which tokenizer "wins" on token count, and is that win meaningful?

**Your answer:**

_(write here)_

## Reproducibility experiment

Is *training* reproducible? Train each algorithm several times on the same corpus and compare the vocabularies. Decide first **how** you will compare two vocabularies.

### 🧩 Exercise 6.7 — Measure vocabulary overlap

Complete the Jaccard similarity of two vocabularies (as *sets of tokens*): |A ∩ B| / |A ∪ B|.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Python sets: `a & b` is the intersection, `a | b` the union, `len(...)` their size.
* 1.0 means identical vocabularies.

In [ ]:
def jaccard(a, b):
    return len(____) / len(____)

runs = {"bpe": [set(train_bpe(CORPUS, 2000).get_vocab()) for _ in range(3)],
        "wordpiece": [set(train_wordpiece(CORPUS, 2000).get_vocab()) for _ in range(3)]}
for name, vocs in runs.items():
    print(name, "Jaccard(run1, run2) =", round(jaccard(vocs[0], vocs[1]), 4), "| Jaccard(run1, run3) =", round(jaccard(vocs[0], vocs[2]), 4))

In [ ]:
assert jaccard({1, 2, 3}, {2, 3, 4}) == 0.5 and jaccard({1}, {1}) == 1.0
print("✅ Exercise 6.7 OK")

**Explain what you see**, and what it implies for how you publish a tokenizer.

**Your answer:**

_(write here)_

## Part F — Save, reload, wrap

First the raw Hugging Face calls (exercises), then the packaged functions of `src/tokenizer_io.py`.

### 🧩 Exercise 6.8 — Save and reload with `tokenizers`

Save your BPE tokenizer from exercise 6.1–6.3 to a JSON file and reload it into a new object; check that both behave the same.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `tokenizer.save(path_as_string)` writes the whole pipeline to JSON.
* `Tokenizer.from_file(path_as_string)` is the class method that loads it.
* Create the folder first: `Path.mkdir(parents=True, exist_ok=True)`.

In [ ]:
from pathlib import Path
scratch = ROOT / "models" / "scratch"; scratch.mkdir(parents=True, exist_ok=True)
path = scratch / "my_bpe.json"

tokenizer.____(str(path))
reloaded = Tokenizer.____(str(path))

s = "Je n'ai pas encore décidé."
print(tokenizer.encode(s).ids == reloaded.encode(s).ids)

In [ ]:
for s in SENTENCES:
    assert tokenizer.encode(s).ids == reloaded.encode(s).ids
print("✅ Exercise 6.8 OK")

In [ ]:
data = json.loads(path.read_text(encoding="utf-8"))
print(list(data.keys()))
print("model type:", data["model"]["type"], "| vocab entries:", len(data["model"]["vocab"]), "| merges:", len(data["model"]["merges"]))
print("pre_tokenizer:", data["pre_tokenizer"]); print("decoder:", data["decoder"])
print("added_tokens:", [(t["id"], t["content"]) for t in data["added_tokens"]])

### 🧩 Exercise 6.9 — Wrap it for Transformers

Wrap the saved file in a `PreTrainedTokenizerFast`, declare its special tokens, batch two sentences with padding, then `save_pretrained` and reload with `AutoTokenizer`.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `PreTrainedTokenizerFast(tokenizer_file=<path string>, unk_token=..., pad_token=..., bos_token=..., eos_token=...)`.
* Call it like any Transformers tokenizer: `fast([...], padding=True)`.
* `fast.save_pretrained(folder)` then `AutoTokenizer.from_pretrained(folder)`.
* Only declare special tokens that exist in your tokenizer (BPE: `[UNK] [PAD] [BOS] [EOS]`).

In [ ]:
from transformers import PreTrainedTokenizerFast, AutoTokenizer

fast = PreTrainedTokenizerFast(
    ____=str(path),
    unk_token="[UNK]", pad_token=____, bos_token="[BOS]", eos_token="[EOS]",
)
batch = fast(["Hello world", "A much longer sentence than the first one"], ____)
print(batch["input_ids"]); print(batch["attention_mask"])

folder = scratch / "my_bpe_hf"
fast.____(str(folder))
again = AutoTokenizer.____(str(folder))
print(again.pad_token, again.pad_token_id)

In [ ]:
assert fast.is_fast and fast.pad_token_id == 1 and again.pad_token == "[PAD]"
assert again.encode("Hello world", add_special_tokens=False) == tokenizer.encode("Hello world").ids
assert len({len(x) for x in batch["input_ids"]}) == 1
print("✅ Exercise 6.9 OK")

## ⏸ Package Part F in `src/tokenizer_io.py`, then continue

`save_tokenizer`, `load_tokenizer`, `to_transformers`, `find_mismatches` → `pytest tests/test_tokenizer_io.py`.

In [ ]:
MODELS = ROOT / "models"
save_tokenizer(bpe, MODELS / "bpe/tokenizer.json"); save_tokenizer(wp, MODELS / "wordpiece/tokenizer.json")
bpe2, wp2 = load_tokenizer(MODELS / "bpe/tokenizer.json"), load_tokenizer(MODELS / "wordpiece/tokenizer.json")
print("BPE mismatches      :", len(find_mismatches(bpe, bpe2, SENTENCES)))
print("WordPiece mismatches:", len(find_mismatches(wp, wp2, SENTENCES)))
fast = to_transformers(MODELS / "bpe/tokenizer.json", unk_token="[UNK]", pad_token="[PAD]", bos_token="[BOS]", eos_token="[EOS]")
print(type(fast).__name__, "| fast:", fast.is_fast, "| vocab:", len(fast))

**Questions (saving)** — 1. why is saving only the vocabulary not enough? 2. what else must be preserved? (use the JSON keys printed above) 3. what is `tokenizer.json`? 4. role of `PreTrainedTokenizerFast`?

Finally write **at least five tests of your own** in `tests/test_my_tokenizers.py` (see the walkthrough, §7.4).

**Your answer:**

_(write here)_